# IN718 Sample 0: SLERP Comparison

This notebook compares three quaternion upsampling modes on **IN718 Test sample 0**:

1. Basic SLERP (no symmetry)
2. Symmetry-aware SLERP
3. Boundary-aware SLERP (labels + SDF guidance)


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from orix.quaternion import symmetry as SYM

candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = None
for p in candidates:
    if (p / "training").exists() and (p / "boundary_aware_slerp.py").exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root from notebook cwd.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from training.quaternion_dataset import QuaternionDataset
from visualization.ipf_render import render_ipf_image, render_ipf_rgb
from boundary_aware_slerp import (
    qnorm,
    slerp,
    symmetrize_pair,
    make_fcc_symmetry_4x4,
    misorientation_deg,
    seam_crossing,
    seam_crossing_heatmap,
)
from boundary_aware_slerp_v2 import (
    SymBilinearSlerpUpsampleV2,
    run_boundary_smoothed_slerp,
    compute_thin_gb_mask,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
dataset_dir = Path("/data/warren/materials/EBSD/IN718_FZ_2D_SR_x4")
sample_idx = 0
scale = 4
out_dir = REPO_ROOT / "outputs" / "notebooks" / "in718_sample0_slerp_compare"
out_dir.mkdir(parents=True, exist_ok=True)

ds = QuaternionDataset(dataset_root=str(dataset_dir), split="Test")
q_lr = ds[sample_idx][0].unsqueeze(0).to(device=device, dtype=torch.float32)
q_lr = qnorm(q_lr)

sym_npy = REPO_ROOT / "symmetry_groups" / "O_group.npy"
if sym_npy.exists():
    sym_ops = torch.tensor(np.load(sym_npy), dtype=q_lr.dtype, device=device)
else:
    sym_ops = make_fcc_symmetry_4x4(device=device, dtype=q_lr.dtype)

sym_class = SYM.O
print("q_lr:", tuple(q_lr.shape))
print("sym_ops:", tuple(sym_ops.shape))
print("out_dir:", out_dir)


In [ ]:
class BasicBilinearSlerpUpsample(nn.Module):
    """Bilinear SLERP without crystal symmetry handling."""

    def __init__(self, scale_factor: int):
        super().__init__()
        self.scale = int(scale_factor)

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        assert C == 4
        s = self.scale
        x = qnorm(x)

        H_out, W_out = H * s, W * s
        device, dtype = x.device, x.dtype

        iy = torch.arange(H_out, device=device, dtype=dtype)
        ix = torch.arange(W_out, device=device, dtype=dtype)
        ys = (iy + 0.5) / s - 0.5
        xs = (ix + 0.5) / s - 0.5

        y0 = torch.floor(ys).clamp(0, H - 1).long()
        x0 = torch.floor(xs).clamp(0, W - 1).long()
        y1 = (y0 + 1).clamp(0, H - 1)
        x1 = (x0 + 1).clamp(0, W - 1)

        v = (ys - y0.to(dtype)).clamp(0.0, 1.0)
        u = (xs - x0.to(dtype)).clamp(0.0, 1.0)
        v_grid, u_grid = torch.meshgrid(v, u, indexing="ij")

        y0g = y0.view(H_out, 1).expand(H_out, W_out)
        y1g = y1.view(H_out, 1).expand(H_out, W_out)
        x0g = x0.view(1, W_out).expand(H_out, W_out)
        x1g = x1.view(1, W_out).expand(H_out, W_out)

        q00 = x[:, :, y0g, x0g]
        q01 = x[:, :, y0g, x1g]
        q10 = x[:, :, y1g, x0g]
        q11 = x[:, :, y1g, x1g]

        def flat(q: torch.Tensor) -> torch.Tensor:
            return q.permute(0, 2, 3, 1).reshape(-1, 4)

        q00_f = flat(q00)
        q01_f = flat(q01)
        q10_f = flat(q10)
        q11_f = flat(q11)

        u_full = u_grid.unsqueeze(0).expand(B, -1, -1).reshape(-1)
        v_full = v_grid.unsqueeze(0).expand(B, -1, -1).reshape(-1)

        q0u = slerp(q00_f, q01_f, u_full)
        q1u = slerp(q10_f, q11_f, u_full)
        q_uv = slerp(q0u, q1u, v_full)

        out = q_uv.view(B, H_out, W_out, 4).permute(0, 3, 1, 2).contiguous()
        return qnorm(out)


def segment_grains_simple(q_lr: torch.Tensor, sym_ops: torch.Tensor, thr_deg: float = 3.0):
    """Simple connectivity segmentation from symmetry-reduced neighbor misorientation."""
    q = qnorm(q_lr)
    _, _, H, W = q.shape

    q0_v = q[:, :, 0 : H - 1, :].permute(0, 2, 3, 1).reshape(-1, 4)
    q1_v = q[:, :, 1:H, :].permute(0, 2, 3, 1).reshape(-1, 4)
    q0_h = q[:, :, :, 0 : W - 1].permute(0, 2, 3, 1).reshape(-1, 4)
    q1_h = q[:, :, :, 1:W].permute(0, 2, 3, 1).reshape(-1, 4)

    ang_v = misorientation_deg(q0_v, q1_v, sym_ops).reshape(H - 1, W).detach().cpu().numpy()
    ang_h = misorientation_deg(q0_h, q1_h, sym_ops).reshape(H, W - 1).detach().cpu().numpy()

    b_v = ang_v > thr_deg
    b_h = ang_h > thr_deg

    n = H * W
    parent = np.arange(n, dtype=np.int64)
    rank = np.zeros(n, dtype=np.int64)

    def find(a: int) -> int:
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    def idx(y: int, x: int) -> int:
        return y * W + x

    for y in range(H):
        for x in range(W - 1):
            if not b_h[y, x]:
                union(idx(y, x), idx(y, x + 1))
    for y in range(H - 1):
        for x in range(W):
            if not b_v[y, x]:
                union(idx(y, x), idx(y + 1, x))

    root_to_lab = {}
    labels = np.empty(n, dtype=np.int64)
    next_lab = 0
    for i in range(n):
        r = find(i)
        if r not in root_to_lab:
            root_to_lab[r] = next_lab
            next_lab += 1
        labels[i] = root_to_lab[r]

    return labels.reshape(H, W), next_lab


def seam_ratio(q: torch.Tensor, sym_ops: torch.Tensor, threshold_deg: float = 5.0):
    q = qnorm(q)
    _, _, H, W = q.shape
    q0_v = q[:, :, 0 : H - 1, :].permute(0, 2, 3, 1).reshape(-1, 4)
    q1_v = q[:, :, 1:H, :].permute(0, 2, 3, 1).reshape(-1, 4)
    q0_h = q[:, :, :, 0 : W - 1].permute(0, 2, 3, 1).reshape(-1, 4)
    q1_h = q[:, :, :, 1:W].permute(0, 2, 3, 1).reshape(-1, 4)

    m_v = seam_crossing(q0_v, q1_v, sym_ops, threshold_deg=threshold_deg)
    m_h = seam_crossing(q0_h, q1_h, sym_ops, threshold_deg=threshold_deg)
    seam_count = int(m_v.sum().item() + m_h.sum().item())
    seam_pairs = int(m_v.numel() + m_h.numel())
    return seam_count / max(seam_pairs, 1), seam_count, seam_pairs


In [ ]:
basic_up = BasicBilinearSlerpUpsample(scale_factor=scale).to(device)
sym_up = SymBilinearSlerpUpsampleV2(
    scale_factor=scale,
    seam_threshold_deg=0.0,
    device=str(device),
    dtype=q_lr.dtype,
).to(device)

with torch.inference_mode():
    q_hr_basic = basic_up(q_lr)
    q_hr_sym = sym_up(q_lr)

labels_lr_np, n_grains = segment_grains_simple(q_lr, sym_ops, thr_deg=3.0)
labels_lr = torch.from_numpy(labels_lr_np).long().to(device)

q_hr_bdry, labels_hr_smooth = run_boundary_smoothed_slerp(
    q_lr=q_lr,
    labels_lr=labels_lr,
    scale=scale,
    upsampler=sym_up,
    smooth_iterations=40,
    smooth_lam=0.15,
    use_sdf=True,
)

print("estimated LR grains:", n_grains)
for name, q in [
    ("basic", q_hr_basic),
    ("symmetry_aware", q_hr_sym),
    ("boundary_aware", q_hr_bdry),
]:
    ratio, seam_count, seam_pairs = seam_ratio(q, sym_ops, threshold_deg=5.0)
    print(f"{name:>15s}: seam={seam_count}/{seam_pairs} ({100.0 * ratio:.3f}%)")


In [ ]:
def q_to_hwc_np(q: torch.Tensor) -> np.ndarray:
    return qnorm(q).squeeze(0).permute(1, 2, 0).detach().cpu().numpy()

q_lr_np = q_to_hwc_np(q_lr)
q_basic_np = q_to_hwc_np(q_hr_basic)
q_sym_np = q_to_hwc_np(q_hr_sym)
q_bdry_np = q_to_hwc_np(q_hr_bdry)

rgbs = {
    "basic": render_ipf_rgb(q_basic_np, sym_class, ref_dir="Z"),
    "symmetry_aware": render_ipf_rgb(q_sym_np, sym_class, ref_dir="Z"),
    "boundary_aware": render_ipf_rgb(q_bdry_np, sym_class, ref_dir="Z"),
}
heats = {
    "basic": seam_crossing_heatmap(torch.from_numpy(q_basic_np).permute(2, 0, 1).unsqueeze(0).to(device), sym_ops).detach().cpu().numpy(),
    "symmetry_aware": seam_crossing_heatmap(torch.from_numpy(q_sym_np).permute(2, 0, 1).unsqueeze(0).to(device), sym_ops).detach().cpu().numpy(),
    "boundary_aware": seam_crossing_heatmap(torch.from_numpy(q_bdry_np).permute(2, 0, 1).unsqueeze(0).to(device), sym_ops).detach().cpu().numpy(),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
names = ["basic", "symmetry_aware", "boundary_aware"]
titles = ["Basic SLERP", "Symmetry-aware SLERP", "Boundary-aware SLERP"]

for i, (name, title) in enumerate(zip(names, titles)):
    axes[0, i].imshow(rgbs[name])
    axes[0, i].set_title(title)
    axes[0, i].axis("off")

    axes[1, i].imshow(heats[name], cmap="hot")
    axes[1, i].set_title(f"Seam Heatmap: {title}")
    axes[1, i].axis("off")

plt.tight_layout()
fig.savefig(out_dir / "comparison_grid.png", dpi=200, bbox_inches="tight")
plt.show()

gb_mask = compute_thin_gb_mask(labels_hr_smooth.detach().cpu().numpy())
overlay = rgbs["boundary_aware"].copy()
overlay[gb_mask] = np.array([0.0, 0.0, 0.0], dtype=overlay.dtype)

plt.figure(figsize=(6, 6))
plt.imshow(np.clip(overlay, 0.0, 1.0))
plt.title("Boundary-aware IPF + Smoothed HR GB")
plt.axis("off")
plt.tight_layout()
plt.savefig(out_dir / "boundary_overlay.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
render_ipf_image(
    q_lr_np,
    sym_class,
    out_png=str(out_dir / "ipf_lr.png"),
    ref_dir="Z",
    include_key=True,
    overwrite=True,
)
render_ipf_image(
    q_basic_np,
    sym_class,
    out_png=str(out_dir / "ipf_basic_slerp_x4.png"),
    ref_dir="Z",
    include_key=True,
    overwrite=True,
)
render_ipf_image(
    q_sym_np,
    sym_class,
    out_png=str(out_dir / "ipf_symmetry_aware_slerp_x4.png"),
    ref_dir="Z",
    include_key=True,
    overwrite=True,
)
render_ipf_image(
    q_bdry_np,
    sym_class,
    out_png=str(out_dir / "ipf_boundary_aware_slerp_x4.png"),
    ref_dir="Z",
    include_key=True,
    overwrite=True,
)
print("Saved files to:", out_dir)
